In [25]:
import importlib.util
import os
import random
import subprocess
import sys
from pathlib import Path
from typing import Optional

if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [6]:
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "data").is_dir():
    REPO_ROOT = Path.cwd().resolve().parent
DATA_ROOT = REPO_ROOT / "data"

BTXRD_ROOT = DATA_ROOT / "BTXRD"
BTXRD_IMAGES = BTXRD_ROOT / "images"
BTXRD_XLSX = BTXRD_ROOT / "dataset.xlsx"
CKPT_DIR = REPO_ROOT / "models" / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BTXRD_MODEL_PATH = CKPT_DIR / "btxrd_from_mura.keras"

IMG_SIZE = 224
BATCH_SIZE = 32

LR = 1e-4

assert BTXRD_IMAGES.is_dir(), f"Missing {BTXRD_IMAGES}"
assert BTXRD_XLSX.is_file(), f"Missing {BTXRD_XLSX}"
print("DATA_ROOT:", DATA_ROOT)

DATA_ROOT: /Users/adyan/Desktop/COSC 4337/Bone-Cancer-Detection/data


In [8]:
def decode_image(path, label, size):
    data = tf.io.read_file(path)
    img = tf.io.decode_image(data, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [size, size])
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.resnet_v2.preprocess_input(img)
    return img, label


def make_dataset(paths, labels, *, size, batch_size, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(paths), 10_000), reshuffle_each_iteration=True)
    ds = ds.map(
        lambda p, y: decode_image(p, y, size),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

mura_train_ds = make_dataset(
    train_paths.tolist(), train_y.tolist(),
    size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True,
)
mura_val_ds = make_dataset(
    val_paths.tolist(), val_y.tolist(),
    size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
)

NameError: name 'train_paths' is not defined

In [7]:
def resolve_image_path(images_dir: Path, image_id: str) -> Optional[Path]:
    p = images_dir / str(image_id).strip()
    if p.is_file():
        return p
    stem = Path(str(image_id)).stem
    for ext in (".jpeg", ".jpg", ".png", ".JPEG", ".JPG", ".PNG"):
        c = images_dir / f"{stem}{ext}"
        if c.is_file():
            return c
    return None

btxrd_df = pd.read_excel(BTXRD_XLSX, engine="openpyxl")
btxrd_df.columns = [str(c).strip() for c in btxrd_df.columns]
if "tumor" not in btxrd_df.columns:
    raise KeyError(f"Expected a 'tumor' column; got: {list(btxrd_df.columns)}")

rows = []
for _, r in btxrd_df.iterrows():
    img_id = r["image_id"]
    p = resolve_image_path(BTXRD_IMAGES, img_id)
    if p is None:
        continue
    rows.append((p.as_posix(), int(r["tumor"])))

btx_paths = [a for a, _ in rows]
btx_labels = np.array([b for _, b in rows], dtype="float32")
print("BTXRD samples with files on disk:", len(btx_paths))
print("Tumor rate:", float(btx_labels.mean()))

b_train_p, b_val_p, b_train_y, b_val_y = train_test_split(
    btx_paths, btx_labels, test_size=0.2, random_state=42, stratify=btx_labels
)
print("BTXRD train / val:", len(b_train_p), len(b_val_p))

BTXRD samples with files on disk: 3746
Tumor rate: 0.4983983039855957
BTXRD train / val: 2996 750


In [9]:
btx_train_ds = make_dataset(
    b_train_p, b_train_y.tolist(),
    size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True,
)
btx_val_ds = make_dataset(
    b_val_p, b_val_y.tolist(),
    size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
)

In [21]:
def build_baseline_model():

    model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Rescaling(1./255),

    tf.keras.layers.Conv2D(32, (5, 5), padding="same", activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    tf.keras.layers.Conv2D(64, (5,5), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),

    tf.keras.layers.Conv2D(128, (5, 5), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
    model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)
    return model

In [22]:
def build_improved_model():
    model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Rescaling(1./255),

    # --- Block 1: Implementing stacked 3x3s with batch norm. ---
    tf.keras.layers.Conv2D(32, (3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    
    tf.keras.layers.Conv2D(32, (3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # --- Block 2: Increase depth. ---
    tf.keras.layers.Conv2D(64, (3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),

    tf.keras.layers.Conv2D(64, (3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # --- Block 3: Dense Head. ---
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
    model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)
    return model

In [23]:
def train_baseline(train_ds, val_ds):
    epochs = [5, 10 , 25, 50]
    baseline_histories = {}
    for epoch in epochs:
        model = build_baseline_model()

        history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epoch,
        verbose=0,
    )
        baseline_histories[epoch] = history.history
        loss = history.history["loss"][-1]
        accuracy = history.history["accuracy"][-1]
        auc = history.history["auc"][-1]
        val_loss = history.history["val_loss"][-1]
        val_accuracy = history.history["val_accuracy"][-1]
        val_auc = history.history["val_auc"][-1]
        print(f"Metrics for {epoch} epochs is:")
        print(f"  Train loss: {loss:.4f}")
        print(f"  Train accuracy: {accuracy:.4f}")
        print(f"  Train AUC: {auc:.4f}")
        print(f"  Val loss: {val_loss:.4f}")
        print(f"  Val accuracy: {val_accuracy:.4f}")
        print(f"  Val AUC: {val_auc:.4f}")
        print()

In [24]:
def train_improved(train_ds, val_ds):
    epochs = [5, 10 , 25, 50]
    improved_histories = {}
    for epoch in epochs:
        model = build_improved_model()

        history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epoch,
        verbose=0,
    )
        improved_histories[epoch] = history.history
        
        loss = history.history["loss"][-1]
        accuracy = history.history["accuracy"][-1]
        auc = history.history["auc"][-1]
        val_loss = history.history["val_loss"][-1]
        val_accuracy = history.history["val_accuracy"][-1]
        val_auc = history.history["val_auc"][-1]
        print(f"Metrics for {epoch} epochs is:")
        print(f"  Train loss: {loss:.4f}")
        print(f"  Train accuracy: {accuracy:.4f}")
        print(f"  Train AUC: {auc:.4f}")
        print(f"  Val loss: {val_loss:.4f}")
        print(f"  Val accuracy: {val_accuracy:.4f}")
        print(f"  Val AUC: {val_auc:.4f}")
        print()

In [26]:
def plot_learning_curves(histories):

    for epoch_count, history in histories.items():
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))

        ax[0].plot(history["loss"], label="Train loss")
        ax[0].plot(history["val_loss"], label="Val loss")
        ax[0].set_title(f"Loss curve: {epoch_count} epochs")
        ax[0].set_xlabel("Epoch")
        ax[0].set_ylabel("Loss")
        ax[0].legend()

        ax[1].plot(history["accuracy"], label="Train accuracy")
        ax[1].plot(history["val_accuracy"], label="Val accuracy")
        ax[1].set_title(f"Accuracy curve: {epoch_count} epochs")
        ax[1].set_xlabel("Epoch")
        ax[1].set_ylabel("Accuracy")
        ax[1].legend()

        plt.show()